# Обучение модели YOLOv8 для Raspberry Pi Zero 2 W (Bookworm 32-bit) + CSI

In [ ]:

# 0) Авто-определение среды
import os, sys, platform
from pathlib import Path

ENV = os.environ.get("ENV", "").lower()  # можно задать: colab / kaggle / local
if not ENV:
    if Path("/content").exists():
        ENV = "colab"
    elif Path("/kaggle").exists():
        ENV = "kaggle"
    else:
        ENV = "local"

print("ENV =", ENV)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())


## 1) Установка зависимостей

In [ ]:

# Установка ultralytics + opencv
# Colab/Kaggle: ставим внутри ноутбука
if ENV in ("colab", "kaggle"):
    !pip -q install "ultralytics==8.*" opencv-python pyyaml matplotlib

# Local: обычно ставят в терминале:
#   pip install ultralytics opencv-python pyyaml matplotlib
# Если хотите — можно раскомментировать:
# if ENV == "local":
#     !pip install "ultralytics==8.*" opencv-python pyyaml matplotlib


## 2) Импорты и проверка

In [ ]:

from ultralytics import YOLO
import cv2
import yaml
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Ultralytics version loaded OK")



## 3) Пути к датасету

Ожидаем структуру (YOLO-format):

```
DATA_ROOT/
  images/
    train/
    val/
  labels/
    train/
    val/
```
- В `images/*` — изображения (`.jpg/.png/...`)
- В `labels/*` — текстовые файлы `.txt` **с тем же stem**, что и у картинки.

Пример:
- `images/train/img_001.jpg`
- `labels/train/img_001.txt`

Формат строки в label-файле:
```
<class_id> <x_center> <y_center> <width> <height>
```
где координаты нормированы в диапазон 0..1.

Ниже выберите, где лежит датасет.


In [ ]:

# 3.1) Укажите DATA_ROOT под вашу среду

DATA_ROOT = None

if ENV == "colab":
    # Вариант 1: датасет уже загружен в /content/dataset
    DATA_ROOT = Path("/content/dataset")

    # Вариант 2: из Google Drive (раскомментируйте)
    # from google.colab import drive
    # drive.mount("/content/drive")
    # DATA_ROOT = Path("/content/drive/MyDrive/datasets/my_yolo_dataset")

elif ENV == "kaggle":
    # Обычно датасеты в /kaggle/input/<dataset-name>/...
    # Посмотрите список папок, затем укажите конкретную:
    base = Path("/kaggle/input")
    print("Kaggle input folders:", [p.name for p in base.iterdir()][:50])
    # Пример:
    # DATA_ROOT = base / "my-yolo-dataset"

else:
    # Локально: положите датасет в ./dataset или укажите путь
    DATA_ROOT = Path("./dataset").resolve()

print("DATA_ROOT =", DATA_ROOT)
assert DATA_ROOT is not None and DATA_ROOT.exists(), f"DATA_ROOT не найден: {DATA_ROOT}"


## 4) Проверка структуры + создание `data.yaml`

In [ ]:

images_train = DATA_ROOT / "images" / "train"
images_val   = DATA_ROOT / "images" / "val"
labels_train = DATA_ROOT / "labels" / "train"
labels_val   = DATA_ROOT / "labels" / "val"

for p in [images_train, images_val, labels_train, labels_val]:
    print(f"{p} -> exists: {p.exists()}")

assert images_train.exists() and labels_train.exists(), "Не найдены папки train images/labels"
assert images_val.exists() and labels_val.exists(), "Не найдены папки val images/labels"

# 4.1) Классы (под ваш кейс: 4 класса — поменяйте названия под себя)
CLASS_NAMES = ["plastic", "paper", "metal", "glass"]  # <-- поменяйте при необходимости

data_yaml = {
    "path": str(DATA_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}

data_yaml_path = DATA_ROOT / "data.yaml"
with open(data_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print("Saved:", data_yaml_path)
print(open(data_yaml_path, "r", encoding="utf-8").read())


## 5) Санити-чек: визуализация примеров с боксами

In [ ]:

def read_yolo_labels(label_path: Path):
    boxes = []
    if not label_path.exists():
        return boxes
    txt = label_path.read_text(encoding="utf-8").strip()
    if not txt:
        return boxes
    for line in txt.splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls, xc, yc, w, h = parts
        boxes.append((int(cls), float(xc), float(yc), float(w), float(h)))
    return boxes

def draw_boxes(img_bgr, boxes, class_names):
    h, w = img_bgr.shape[:2]
    out = img_bgr.copy()
    for cls, xc, yc, bw, bh in boxes:
        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w-1, x2), min(h-1, y2)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0,255,0), 2)
        name = class_names[cls] if 0 <= cls < len(class_names) else str(cls)
        cv2.putText(out, name, (x1, max(0, y1-6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    return out

train_imgs = sorted(list(images_train.glob("*.*")))
assert len(train_imgs) > 0, "Нет изображений в images/train"

N = 3
sample = random.sample(train_imgs, min(N, len(train_imgs)))

for img_path in sample:
    label_path = labels_train / (img_path.stem + ".txt")
    img = cv2.imread(str(img_path))
    assert img is not None, f"Не удалось прочитать изображение: {img_path}"
    boxes = read_yolo_labels(label_path)
    vis = draw_boxes(img, boxes, CLASS_NAMES)
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(9, 6))
    plt.title(img_path.name)
    plt.axis("off")
    plt.imshow(vis_rgb)
    plt.show()



## 6) Обучение (настройки под edge/Zero 2 W)

Рекомендации под последующий запуск на Raspberry Pi Zero 2 W:
- берём **yolov8n** (nano)
- обучаем при `imgsz=640` для качества **или** `imgsz=416/320` если датасет простой и нужна скорость
- на Zero 2 W чаще всего инференс будет на `imgsz=320` (или 256)

Ниже параметры можно менять.


In [ ]:

MODEL_NAME = "yolov8n.pt"   # стартовая nano-модель
EPOCHS = 80                # начните с 50-100
IMGSZ_TRAIN = 640          # 640 для качества; можно 416/320 для более edge-ориентированного обучения
BATCH = 16                 # Colab GPU: 16-32; CPU: 4-8
WORKERS = 2

# device:
# - Colab/Kaggle GPU: device=0
# - CPU: device="cpu"
DEVICE = 0
if ENV == "local":
    # если локально без GPU — поставим CPU
    DEVICE = "cpu"

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ_TRAIN,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project="runs_train",
    name="exp_zero2w",
    patience=20,          # early stopping
    close_mosaic=10       # ближе к концу отключаем mosaic
)

print("Train done.")


## 7) Валидация + путь к лучшим весам

In [ ]:

best_pt = Path("runs_train/exp_zero2w/weights/best.pt")
last_pt = Path("runs_train/exp_zero2w/weights/last.pt")

print("best:", best_pt, "exists:", best_pt.exists())
print("last:", last_pt, "exists:", last_pt.exists())

assert best_pt.exists(), "Не найден best.pt — проверьте, что обучение завершилось и папка runs_train доступна."

trained = YOLO(str(best_pt))

metrics = trained.val(
    data=str(data_yaml_path),
    imgsz=IMGSZ_TRAIN,
    device=DEVICE,
)
print(metrics)


## 8) Инференс-пример (сохранит картинки с предсказаниями)

In [ ]:

# Прогон по val-картинкам и сохранение результатов в runs_predict/...
pred_out = trained.predict(
    source=str(images_val),
    imgsz=320,       # как будет ближе к Raspberry
    conf=0.35,
    device=DEVICE,
    save=True,
    project="runs_predict",
    name="val_pred_zero2w",
    verbose=False
)
print("Saved predictions to runs_predict/val_pred_zero2w")



## 9) Экспорт для Raspberry Pi Zero 2 W

Для Bookworm 32-bit и слабого CPU чаще всего практично:
- **TFLite** (FP16) — быстро и просто
- **TFLite INT8** — ещё быстрее, но нужен representative dataset (калибровка)

Ниже экспортируем оба варианта.


In [ ]:

# 9.1) Экспорт в TFLite (FP16). Обычно проще всего.
tflite_fp16_path = trained.export(format="tflite", imgsz=320, half=True)
print("TFLite FP16 exported:", tflite_fp16_path)



### 9.2) Экспорт в TFLite INT8 (калибровка)

INT8 обычно даёт лучший шанс на скорость на Zero 2 W.

Ultralytics попросит representative dataset. Мы дадим генератор, который:
- берёт N изображений из `images/train`
- ресайзит до `imgsz` (320)
- отдаёт как float32 в диапазоне 0..1 (как обычно для калибровки)

> Калибровку можно делать на 100–500 изображениях.


In [ ]:

# Representative dataset generator for INT8 export
CALIB_IMAGES = 200   # 100-500 обычно достаточно
IMGSZ_EXPORT = 320

train_img_list = sorted(list(images_train.glob("*.*")))
assert len(train_img_list) > 0, "Нет изображений для калибровки"

# Ограничим количество
calib_list = train_img_list[:min(CALIB_IMAGES, len(train_img_list))]

def representative_data_gen():
    for p in calib_list:
        img = cv2.imread(str(p))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMGSZ_EXPORT, IMGSZ_EXPORT), interpolation=cv2.INTER_LINEAR)
        img = img.astype(np.float32) / 255.0
        img = np.expand_dims(img, axis=0)
        yield [img]

# Экспорт INT8
tflite_int8_path = trained.export(
    format="tflite",
    imgsz=IMGSZ_EXPORT,
    int8=True,
    data=str(data_yaml_path),   # полезно для метаданных
    # Ultralytics использует rep dataset внутри; для некоторых версий он берётся из data=...
)

print("TFLite INT8 export attempted. Check output above and the returned path if available.")



## 10) Упаковка артефактов для скачивания

Соберём в `artifacts_zero2w.zip`:
- `best.pt`
- `data.yaml`
- экспортированные модели (если есть)
- несколько примеров предсказаний (опционально)

На Colab: файл появится в файловом браузере слева.  
На Kaggle: появится в `/kaggle/working`.  
Локально: в текущей папке.


In [ ]:

import zipfile

artifacts = []
# веса
if best_pt.exists():
    artifacts.append(best_pt)
# yaml
if data_yaml_path.exists():
    artifacts.append(data_yaml_path)

# Найдём tflite модели в стандартной папке Ultralytics
# Обычно export кладёт рядом в runs/train/exp... или в текущую директорию — поэтому ищем по расширению.
for root in [Path("."), Path("runs_train"), Path("runs_predict")]:
    if root.exists():
        artifacts.extend(root.rglob("*.tflite"))

# (опционально) добавить пару предсказанных картинок
pred_dir = Path("runs_predict/val_pred_zero2w")
if pred_dir.exists():
    # добавим первые 10 файлов картинок
    imgs = list(pred_dir.rglob("*.jpg")) + list(pred_dir.rglob("*.png"))
    artifacts.extend(imgs[:10])

zip_path = Path("artifacts_zero2w.zip").resolve()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:
        p = Path(p)
        if p.exists() and p.is_file():
            z.write(p, arcname=str(p))
print("Created:", zip_path)
print("Files included:", len([p for p in artifacts if Path(p).exists()]))



## 11) (Опционально) Скачать zip в Colab

Если вы в Colab — можно скачать прямо из ноутбука.


In [ ]:

if ENV == "colab":
    from google.colab import files
    files.download("artifacts_zero2w.zip")
